# Stage 6a - Continuous Pose over Rally Segments

Stage 2 extracted only **windowed** pose - 97 frames around each *known* stroke. A contact detector cannot train on that: it would only ever see frames where a stroke exists, and would learn nothing about the gaps between them.

This notebook extracts **continuous** pose over rally segments: every labelled contact, padded and merged, so the sequence includes the between-stroke frames where a detector actually produces its false positives.

### Why segments rather than the whole video

Full continuous extraction over all 644,199 frames costs ~5.7 h. The frames that matter for training are the ones *inside and around rallies*; long dead stretches - ball pickups, towel breaks, scoreboard shots - are what the Phase 2 rally gate filters anyway.

**Cell 3 computes actual coverage and cost before extracting anything.** My earlier "~2 h" estimate assumed +/-3 s padding without checking, so pick the padding from real numbers rather than that guess.

### One thing the segments handle for free

`game_5`'s strokes stop at frame 80,355 of 144,000 - the last 8 minutes are unlabelled and may contain unannotated strokes. Because segments are built *around labelled contacts*, that tail is excluded automatically. Training on it would have produced false negatives.

### Outputs

| file | contents |
|---|---|
| `derived/pose_stream/{video_id}.npz` | per-frame keypoints, scores, boxes, frame indices |
| `derived/meta/segments.json` | segment boundaries per video |
| `derived/meta/stream_quality.csv` | coverage and confidence report |

GPU required. Checkpointed per video - a disconnect costs at most one video.


## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Setup

In [2]:
BASE = "/content/drive/MyDrive/tt_coach"

DET_STRIDE = 8       # detector every N frames, boxes interpolated between
DET_CONF   = 0.35
BOX_PAD    = 0.18
COPY_LOCAL = True    # random access over the Drive FUSE mount is very slow
SKIP_EXISTING = True

!pip install -q --no-deps rtmlib 2>&1 | tail -1
!pip install -q onnxruntime-gpu==1.22.0 ultralytics 2>&1 | tail -1

import os, json, shutil, time, glob, site
from pathlib import Path
import numpy as np, pandas as pd
from tqdm.auto import tqdm

libs = []
for sp in site.getsitepackages():
    libs += glob.glob(os.path.join(sp, "nvidia", "*", "lib"))
if libs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(libs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

import cv2, torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4."

BASE   = Path(BASE); META = BASE/"derived/meta"; VID = BASE/"raw/videos"
STREAM = BASE/"derived/pose_stream"; STREAM.mkdir(parents=True, exist_ok=True)
LOCAL  = Path("/content/_work"); LOCAL.mkdir(exist_ok=True)

def load(stem):
    p = META/f"{stem}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(META/f"{stem}.csv")

strokes = load("strokes")
events  = load("events")
manifest = pd.read_csv(META/"video_manifest.csv").set_index("video_id")
NFRAMES = manifest["nb_frames"].to_dict()

VIDEO_IDS = sorted(strokes.video_id.unique(),
                   key=lambda v: (v.split("_")[0], int(v.split("_")[1])))
FPS = 120
MS_PER_INFER = 0.016      # measured in Stage 2: RTMPose-l on crops, T4

# every racket-ball contact: typed strokes + untyped empty_event markers
contacts = {}
for v in VIDEO_IDS:
    a = strokes.loc[strokes.video_id == v, "frame_120"].values
    b = events.loc[(events.video_id == v) & (events.event == "empty_event"),
                   "frame_120"].values
    contacts[v] = np.unique(np.concatenate([a, b]).astype(int))

print(f"{sum(len(c) for c in contacts.values())} contacts "
      f"({len(strokes)} typed + "
      f"{int((events.event=='empty_event').sum())} empty_event)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.1 MB/s eta 0:00:00
1576 contacts (1457 typed + 119 empty_event)


## 3. Plan before extracting

Coverage is not linear in padding: strokes inside a rally are ~1 s apart, so their windows merge into rally-length segments almost immediately. Past a point, extra padding buys only dead time at full cost.

Read the table, then set `PAD_S` in the next cell.

In [5]:
def build_segments(pad_frames, vids=VIDEO_IDS):
    """Pad every contact by +/- pad_frames, merge overlaps, clip to video."""
    segs = {}
    for v in vids:
        n = NFRAMES[v]
        iv = [(int(max(0, f - pad_frames)), int(min(n - 1, f + pad_frames)))
              for f in contacts[v]]
        merged = []
        for s, e in sorted(iv):
            if merged and s <= merged[-1][1] + 1:
                merged[-1][1] = max(merged[-1][1], e)
            else:
                merged.append([s, e])
        segs[v] = merged
    return segs


rows = []
for pad_s in (0.75, 1.0, 1.5, 2.0, 3.0):
    segs = build_segments(int(pad_s * FPS))
    fr = sum(e - s + 1 for v in VIDEO_IDS for s, e in segs[v])
    total = sum(NFRAMES[v] for v in VIDEO_IDS)
    rows.append({
        "pad_s": pad_s,
        "segments": sum(len(segs[v]) for v in VIDEO_IDS),
        "frames": fr,
        "coverage": fr / total,
        "hours": fr * 2 * MS_PER_INFER / 3600,     # 2 players per frame
        "gb_npz": fr * 2 * 220 / 1e9,
    })
plan = pd.DataFrame(rows)
print(f"total frames across 12 videos: {sum(NFRAMES[v] for v in VIDEO_IDS):,}")
print(f"full continuous extraction   : "
      f"{sum(NFRAMES[v] for v in VIDEO_IDS)*2*MS_PER_INFER/3600:.1f} h\n")
print(plan.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("""
  Choosing:
  - The negatives that matter are the frames BETWEEN strokes inside a rally.
     Those are captured at any padding, because consecutive strokes are ~1 s
     apart and their windows merge regardless.
  - Extra padding buys pre-serve and post-point dead time. Some is useful
     (a detector must stay quiet through it); a lot is just cost.
  - 1.0-1.5 s is usually the knee of the curve. Prefer the smallest padding
     whose coverage still leaves clear gaps between segments.""")

total frames across 12 videos: 644,199
full continuous extraction   : 5.7 h

 pad_s  segments  frames  coverage  hours  gb_npz
 0.750       364  159386     0.247  1.417   0.070
 1.000       347  180516     0.280  1.605   0.079
 1.500       328  220292     0.342  1.958   0.097
 2.000       316  258760     0.402  2.300   0.114
 3.000       281  329255     0.511  2.927   0.145

  Choosing:
   - The negatives that matter are the frames BETWEEN strokes inside a rally.
     Those are captured at any padding, because consecutive strokes are ~1 s
     apart and their windows merge regardless.
   - Extra padding buys pre-serve and post-point dead time. Some is useful
     (a detector must stay quiet through it); a lot is just cost.
   - 1.0-1.5 s is usually the knee of the curve. Prefer the smallest padding
     whose coverage still leaves clear gaps between segments.


## 4. Choose padding & build segments

In [6]:
PAD_S = 1.5          # <-- set from the table above

PAD = int(PAD_S * FPS)
SEGMENTS = build_segments(PAD)

rows = []
for v in VIDEO_IDS:
    fr = sum(e - s + 1 for s, e in SEGMENTS[v])
    rows.append({"video_id": v, "contacts": len(contacts[v]),
                 "segments": len(SEGMENTS[v]), "frames": fr,
                 "video_frames": NFRAMES[v], "coverage": fr / NFRAMES[v],
                 "mean_seg_s": fr / max(len(SEGMENTS[v]), 1) / FPS})
seg_df = pd.DataFrame(rows)
print(seg_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

tot = int(seg_df.frames.sum())
print(f"\n  padding    : +/- {PAD_S}s ({PAD} frames)")
print(f"  segments   : {int(seg_df.segments.sum())}")
print(f"  frames     : {tot:,}  ({tot/seg_df.video_frames.sum():.1%} of all video)")
print(f"  inferences : {tot*2:,}  ->  ~{tot*2*MS_PER_INFER/3600:.1f} h")

(META/"segments.json").write_text(json.dumps(
    {"pad_s": PAD_S, "pad_frames": PAD, "fps": FPS,
     "segments": {v: SEGMENTS[v] for v in VIDEO_IDS},
     "n_frames": tot}, indent=2))
print(f"\n-> {META/'segments.json'}")

video_id  contacts  segments  frames  video_frames  coverage  mean_seg_s
  game_1       180        28   20144         88599     0.227       5.995
  game_2       468       102   70873        172200     0.412       5.790
  game_3       167        48   27911         74160     0.376       4.846
  game_4       173        30   24249         63120     0.384       6.736
  game_5       252        56   34004        144000     0.236       5.060
  test_1        85        10    8027         16800     0.478       6.689
  test_2        29         3    2806          3600     0.779       7.794
  test_3        29         5    4070          7800     0.522       6.783
  test_4        76        23   13251         36000     0.368       4.801
  test_5        29         7    3862         11520     0.335       4.598
  test_6        39         8    5335         10800     0.494       5.557
  test_7        49         8    5760         15600     0.369       6.000

  padding    : +/- 1.5s (180 frames)
  segments   

## 5. Load models

In [7]:
import onnxruntime as ort
provs = ort.get_available_providers()
print(f"onnxruntime {ort.__version__}: {provs}")
assert "CUDAExecutionProvider" in provs, (
    "CUDAExecutionProvider missing. Restart the session after cell 2 and re-run.")

from ultralytics import YOLO
from rtmlib import RTMPose

det = YOLO(str(BASE/"models/detector/best.pt")); det.to("cuda")
PLAYER_CLS = [k for k, v in det.names.items() if v.lower() == "player"][0]
TABLE_CLS  = [k for k, v in det.names.items() if v.lower() == "table"][0]

RTM_URL = ("https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/"
           "onnx_sdk/rtmpose-l_simcc-body7_pt-body7_420e-384x288-"
           "3f5a1437_20230504.zip")
pose = RTMPose(onnx_model=RTM_URL, model_input_size=(288, 384),
               backend="onnxruntime", device="cuda")
print("RTMPose-l ready")

onnxruntime 1.22.0: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip" to /root/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip
100%|██████████| 98.9M/98.9M [00:03<00:00, 31.6MB/s]


load /root/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.onnx with onnxruntime backend
RTMPose-l ready


## 6. Extraction

Frames are decoded **sequentially** within each segment after a single seek - seeking a 120 fps h264 file is expensive, sequential reads are not. That is the dominant performance factor here, as it was in Stage 2.

Boxes come from the detector every 8th frame and are linearly interpolated between, which keeps a tight tracking box for ~5% of the naive detection cost.

In [8]:
def detect_players(frames, mid_x):
    out = []
    for r in det.predict(frames, verbose=False, conf=DET_CONF):
        d = {"left": None, "right": None}
        if r.boxes is not None and len(r.boxes):
            xyxy = r.boxes.xyxy.cpu().numpy()
            cls  = r.boxes.cls.cpu().numpy().astype(int)
            pl = xyxy[cls == PLAYER_CLS]
            if len(pl):
                cx = (pl[:, 0] + pl[:, 2]) / 2
                ls, rs = pl[cx < mid_x], pl[cx >= mid_x]
                if len(ls): d["left"]  = ls[np.argmin((ls[:,0]+ls[:,2])/2)]
                if len(rs): d["right"] = rs[np.argmax((rs[:,0]+rs[:,2])/2)]
        out.append(d)
    return out


def interp(dets, idxs, n):
    known = [(i, b) for i, b in zip(idxs, dets) if b is not None]
    if not known:
        return None
    ki = np.array([k[0] for k in known], float)
    kb = np.stack([k[1] for k in known]).astype(float)
    return np.stack([np.interp(np.arange(n), ki, kb[:, c])
                     for c in range(4)], 1).astype(np.float32)


def pad_box(b, W, H, p=BOX_PAD):
    x1, y1, x2, y2 = b; w, h = x2 - x1, y2 - y1
    return np.array([max(0, x1-w*p), max(0, y1-h*p*0.6),
                     min(W, x2+w*p), min(H, y2+h*p*0.25)], np.float32)


def find_table(cap, n, k=9):
    boxes = []
    for f in np.linspace(n*0.1, n*0.9, k).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(f))
        ok, fr = cap.read()
        if not ok: continue
        r = det.predict(fr, verbose=False, conf=DET_CONF)[0]
        if r.boxes is None or not len(r.boxes): continue
        xyxy = r.boxes.xyxy.cpu().numpy(); cls = r.boxes.cls.cpu().numpy().astype(int)
        tb = xyxy[cls == TABLE_CLS]
        if len(tb):
            boxes.append(tb[np.argmax((tb[:,2]-tb[:,0])*(tb[:,3]-tb[:,1]))])
    return np.median(np.stack(boxes), 0).astype(np.float32) if boxes else None


CHUNK = 600      # frames held in memory at once


def extract(vid):
    out = STREAM/f"{vid}.npz"
    if SKIP_EXISTING and out.exists():
        print(f"  {vid}: exists, skipped"); return

    src = VID/f"{vid}.mp4"
    path = src
    if COPY_LOCAL:
        path = LOCAL/f"{vid}.mp4"
        if not path.exists():
            t0 = time.time(); shutil.copy(src, path)
            print(f"  {vid}: copied {src.stat().st_size/1e9:.1f} GB in "
                  f"{time.time()-t0:.0f}s")

    cap = cv2.VideoCapture(str(path))
    nfr = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    table = find_table(cap, nfr)
    mid_x = (table[0]+table[2])/2 if table is not None else W/2

    total = sum(e-s+1 for s, e in SEGMENTS[vid])
    F_IDX = np.zeros(total, np.int32)
    KP = np.zeros((total, 2, 17, 2), np.float16)
    SC = np.zeros((total, 2, 17), np.float16)
    BX = np.zeros((total, 2, 4), np.float16)
    DET = np.zeros((total, 2), bool)
    SEG_ID = np.zeros(total, np.int32)

    w = 0
    pbar = tqdm(total=total, desc=f"  {vid}", leave=False)
    for si, (s0, e0) in enumerate(SEGMENTS[vid]):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(s0))
        pos = s0
        while pos <= e0:
            n = min(CHUNK, e0 - pos + 1)
            frames = []
            for _ in range(n):
                ok, fr = cap.read()
                if not ok:
                    fr = frames[-1] if frames else np.zeros((H, W, 3), np.uint8)
                frames.append(fr)
            n = len(frames)

            di = list(range(0, n, DET_STRIDE))
            if di[-1] != n-1: di.append(n-1)
            dets = detect_players([frames[i] for i in di], mid_x)

            boxes = {}
            for pi, side in enumerate(["left", "right"]):
                b = interp([d[side] for d in dets], di, n)
                if b is not None:
                    boxes[pi] = np.stack([pad_box(x, W, H) for x in b])

            for k in range(n):
                bb, who = [], []
                for pi in (0, 1):
                    if pi in boxes:
                        bb.append(boxes[pi][k]); who.append(pi)
                        BX[w+k, pi] = boxes[pi][k]; DET[w+k, pi] = True
                if bb:
                    kp, sc = pose(frames[k], bboxes=np.stack(bb))
                    for j, pi in enumerate(who):
                        KP[w+k, pi] = kp[j]; SC[w+k, pi] = sc[j]
                F_IDX[w+k] = pos + k
                SEG_ID[w+k] = si

            w += n; pos += n
            pbar.update(n)
            del frames
    pbar.close()
    cap.release()
    if COPY_LOCAL and path != src:
        path.unlink(missing_ok=True)

    lab = np.isin(F_IDX, contacts[vid])
    np.savez_compressed(
        out, frame_idx=F_IDX[:w], seg_id=SEG_ID[:w],
        keypoints=KP[:w], scores=SC[:w], boxes=BX[:w], detected=DET[:w],
        is_contact=lab[:w],
        table_box=table if table is not None else np.zeros(4, np.float32),
        segments=np.array(SEGMENTS[vid]), pad_frames=PAD)
    wr = SC[:w][..., [9, 10]]
    print(f"  {vid}: {w:,} frames  {int(lab[:w].sum())} contacts  "
          f"det={DET[:w].mean():.1%}  wrist={wr[wr>0].mean():.3f}  "
          f"-> {out.stat().st_size/1e6:.0f} MB")


t0 = time.time()
for v in VIDEO_IDS:
    extract(v)
print(f"\ntotal {(time.time()-t0)/60:.1f} min")

  game_1: copied 5.6 GB in 128s


  game_1:   0%|          | 0/20144 [00:00<?, ?it/s]

  game_1: 20,144 frames  180 contacts  det=99.0%  wrist=0.771  -> 3 MB
  game_2: copied 10.8 GB in 381s


  game_2:   0%|          | 0/70873 [00:00<?, ?it/s]

  game_2: 70,873 frames  468 contacts  det=97.2%  wrist=0.719  -> 11 MB
  game_3: copied 4.6 GB in 135s


...

  test_5: 3,862 frames  29 contacts  det=100.0%  wrist=0.753  -> 1 MB
  test_6: copied 0.7 GB in 17s


  test_6:   0%|          | 0/5335 [00:00<?, ?it/s]

  test_6: 5,335 frames  39 contacts  det=92.4%  wrist=0.734  -> 1 MB
  test_7: copied 0.5 GB in 27s


  test_7:   0%|          | 0/5760 [00:00<?, ?it/s]

  test_7: 5,760 frames  49 contacts  det=100.0%  wrist=0.763  -> 1 MB

total 188.2 min


## 7. Quality report

In [9]:
rows = []
for v in VIDEO_IDS:
    p = STREAM/f"{v}.npz"
    if not p.exists(): continue
    d = np.load(p, allow_pickle=True)
    sc, det_, lab = d["scores"], d["detected"], d["is_contact"]
    wr = sc[..., [9, 10]]
    n = len(lab)
    rows.append({
        "video_id": v, "frames": n,
        "contacts_found": int(lab.sum()),
        "contacts_expected": len(contacts[v]),
        "pos_rate": lab.mean(),
        "detected": float(det_.mean()),
        "wrist_conf": float(wr[wr > 0].mean()) if (wr > 0).any() else 0.0,
        "segments": int(d["seg_id"].max() + 1),
    })
q = pd.DataFrame(rows)
q.to_csv(META/"stream_quality.csv", index=False)
print(q.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

miss = q[q.contacts_found != q.contacts_expected]
print("\n" + "=" * 70)
print(f"  frames        : {int(q.frames.sum()):,}")
print(f"  contacts      : {int(q.contacts_found.sum())} / "
      f"{int(q.contacts_expected.sum())}")
print(f"  positive rate : {q.contacts_found.sum()/q.frames.sum():.5f}  "
      f"(1 in {int(q.frames.sum()/max(q.contacts_found.sum(),1))} frames)")
print(f"  wrist conf    : {q.wrist_conf.mean():.3f}")
print(f"  detected      : {q.detected.mean():.1%}")
if len(miss):
    print(f"\n  !! contact count mismatch: {list(miss.video_id)}")
    print("     every contact must fall inside a segment - check cell 4")
else:
    print("\n  All contacts fall inside segments.")
print("""
  The positive rate is the key number for Stage 6b: contacts are ~1 in 200
  frames, so the detector trains on a severely imbalanced per-frame target.
  That is handled there with label smoothing around each contact and a
  positive class weight - not by resampling, which would destroy the
  temporal continuity the model needs.""")
print("=" * 70)

video_id  frames  contacts_found  contacts_expected  pos_rate  detected  wrist_conf  segments
  game_1   20144             180                180    0.0089    0.9896      0.7705        28
  game_2   70873             468                468    0.0066    0.9721      0.7192       102
  game_3   27911             167                167    0.0060    0.9932      0.7920        48
  game_4   24249             173                173    0.0071    0.9771      0.7729        30
  game_5   34004             252                252    0.0074    0.9999      0.8169        56
  test_1    8027              85                 85    0.0106    0.9497      0.7837        10
  test_2    2806              29                 29    0.0103    0.9961      0.7817         3
  test_3    4070              29                 29    0.0071    0.9263      0.7422         5
  test_4   13251              76                 76    0.0057    0.9867      0.7534        23
  test_5    3862              29                 29    0.007

---
## Done

| artifact | used by |
|---|---|
| `derived/pose_stream/*.npz` | Stage 6b - contact detector training |
| `derived/meta/segments.json` | segment boundaries, for decoding and evaluation |
| `derived/meta/stream_quality.csv` | quality reference |

**Confirm before Stage 6b:**
1. All contacts fall inside segments (no mismatch warning)
2. Wrist confidence comparable to Stage 2's 0.804
3. Detection rate > 95%

Next: **`06b_contact_detector.ipynb`** - per-frame contact probability, peak-picking with NMS, target F1 0.85-0.92 at +/-5 frames.
